In [1]:
"""
================================================================================
Проект: Автоматизация обработки данных ионной хроматографии (Ion Balance Automation)
Скрипт: Шаг 2 - Математический расчет ионного баланса (Ion Balance Calculator)
================================================================================

Описание задачи:
Скрипт берет очищенные данные (clean_data.xlsx), где для каждой пробы может быть 
несколько значений концентрации (с разных разбавлений и усилений). 
Используя метод комбинаторики, скрипт перебирает все возможные комбинации, 
чтобы найти ту, при которой ионный баланс сходится с погрешностью < 3%.
Если натрий не измерялся (или его данные не сходятся), он рассчитывается 
методом "по разности" на основе закона электронейтральности.

Входные данные:
- Файл clean_data.xlsx (результат работы скрипта 01_convert_data.py).

Выходные данные:
- Final_Balance_*.xlsx (пробы с успешно рассчитанным балансом).
- Errors_and_Recommendations_*.xlsx (пробы с физическим дисбалансом).
================================================================================
"""

import pandas as pd        # Библиотека для работы с таблицами (DataFrame) и Excel
import numpy as np         # Библиотека для математических операций (не используется напрямую, но полезна)
import itertools           # Модуль для генерации комбинаций (декартово произведение)
import datetime            # Модуль для получения текущей даты и времени (для версионирования файлов)

# 1. Справочник эквивалентных весов (г/моль)
# Для однозарядных ионов вес совпадает с молярной массой, 
# для многозарядных (Ca, Mg, SO4, CO3) — делим молярную массу на заряд.
weights = {
    'F': 19.00, 'Cl': 35.45, 'NO2': 46.01, 'Br': 79.90, 'NO3': 62.00, 
    'PO4': 31.66, 'SO4': 48.03, 'Na': 22.99, 'NH4': 18.04, 'K': 39.10, 
    'Mg': 12.155, 'Ca': 20.04, 'CO3': 30.005, 'HCO3': 61.02
}

# Списки ионов для удобства перебора
anions = ['F', 'Cl', 'NO2', 'Br', 'NO3', 'PO4', 'SO4', 'CO3', 'HCO3']
cations = ['Na', 'NH4', 'K', 'Mg', 'Ca']

# --- НАСТРОЙКА ИСКЛЮЧЕНИЙ ---
# Список ионов, отсутствие которых в пробе не считается ошибкой 
# (например, если фосфаты и аммоний не входят в стандартный перечень анализа вашей лаборатории)
exclusions = ['PO4', 'NO2', 'CO3', 'Br', 'NH4'] 

def calculate_balance(file_name):
    """
    Основная функция для расчета ионного баланса.
    Перебирает все возможные комбинации разбавлений/усилений для поиска 
    наиболее физически достоверного результата (с погрешностью < 3%).
    """
    print("Загрузка очищенных данных...")
    try:
        df = pd.read_excel(file_name)
    except FileNotFoundError:
        print(f"Ошибка: Файл {file_name} не найден.")
        return

    # Генерируем метку времени для версионирования итоговых файлов
    now = datetime.datetime.now()
    timestamp = now.strftime("%Y-%m-%d_%H-%M-%S") 
    file_balance = f'Final_Balance_{timestamp}.xlsx'
    file_errors = f'Errors_and_Recommendations_{timestamp}.xlsx'

    samples = df['Sample'].unique()
    final_results = []
    error_results = []
    
    print(f"Найдено проб: {len(samples)}. Начинаем перебор комбинаций...")

    for sample_id in samples:
        sample_df = df[df['Sample'] == sample_id]
        
        # Собираем ВСЕ доступные значения для каждого иона в данной пробе
        available_data = {}
        for ion in weights.keys():
            ion_data = sample_df[sample_df['Ion'] == ion]
            vals = []
            for _, row in ion_data.iterrows():
                vals.append({'Conc': row['Conc'], 'Gain': row['Gain'], 'Dilution': row['Dilution']})
            
            # Убираем дубликаты концентраций, чтобы не тратить время на одинаковые переборы
            unique_vals = []
            seen_concs = set()
            for v in vals:
                if v['Conc'] not in seen_concs:
                    unique_vals.append(v)
                    seen_concs.add(v['Conc'])
                  
            # --- ЗАЩИТА ОТ ПОДГОНКИ ПОД НУЛИ ---
            # Если у иона есть реальные значения (> 0), мы отбрасываем нули, 
            # чтобы скрипт не мог "схитрить" и обнулить катионы ради баланса.
            real_vals = [v for v in unique_vals if v['Conc'] > 0]
            if real_vals:
                unique_vals = real_vals
            # -----------------------------------
            
            # Если данных нет (прочерк), ставим 0 для математики
            available_data[ion] = unique_vals if unique_vals else [{'Conc': 0, 'Gain': '-', 'Dilution': '-'}]
                
        anion_pools = [available_data[ion] for ion in anions]
        cation_pools = [available_data[ion] for ion in cations]
        
        best_balance = None
        min_error = float('inf')

        # --- МАТЕМАТИЧЕСКИЙ ПЕРЕБОР (ITERTTOOLS.PRODUCT) ---
        # Перебираем декартово произведение всех вариантов анионов и катионов.
        # Это значит, что мы берем каждую возможную комбинацию значений.
        for a_combo in itertools.product(*anion_pools):
            for c_combo in itertools.product(*cation_pools):
                
                # Суммируем анионы в мг-экв/л
                sum_anions = sum(item['Conc'] / weights[ion] for ion, item in zip(anions, a_combo))
                if sum_anions == 0: continue # Пропускаем пробу, если анионов вообще нет
                
                # Суммируем известные катионы (без натрия)
                sum_cations_known = sum(item['Conc'] / weights[ion] for ion, item in zip(cations, c_combo) if ion != 'Na')
                
                # Вариант 1: Проверяем сходимость баланса с ИЗМЕРЕННЫМ натрием
                na_item = c_combo[cations.index('Na')]
                if na_item['Conc'] > 0:
                    sum_cations_total = sum_cations_known + (na_item['Conc'] / weights['Na'])
                    # Погрешность по формуле из классической гидрохимии
                    error = abs(sum_anions - sum_cations_total) / (sum_anions + sum_cations_total) * 100
                    
                    if error < 3.0 and error < min_error: # 3% - допустимая погрешность
                        min_error = error
                        best_balance = {
                            'type': 'measured_Na', 'a_combo': a_combo, 'c_combo': c_combo,
                            'na_item': na_item, 'na_meq': na_item['Conc'] / weights['Na'], 
                            'sum_anions': sum_anions, 'sum_cations': sum_cations_total
                        }
                
                # Вариант 2: Если натрий не сходится, считаем его "по разности" (Закон электронейтральности)
                na_diff_meq = sum_anions - sum_cations_known
                if na_diff_meq > 0: # Натрий не может быть отрицательным
                    # Если мы еще не нашли ничего лучшего, или если вариант с приборным натрием дал плохой баланс
                    if best_balance is None or best_balance['type'] == 'measured_Na' and min_error > 1.0:
                        min_error = 0.0
                        best_balance = {
                            'type': 'diff_Na', 'a_combo': a_combo, 'c_combo': c_combo,
                            'na_item': {'Conc': na_diff_meq * weights['Na'], 'Gain': 'расчет', 'Dilution': '-'},
                            'na_meq': na_diff_meq,
                            'sum_anions': sum_anions, 'sum_cations': sum_cations_known + na_diff_meq
                        }

        # --- ФОРМИРОВАНИЕ ОТЧЕТА ДЛЯ УСПЕШНОЙ ПРОБЫ ---
        if best_balance is not None:
            a_dict = dict(zip(anions, best_balance['a_combo']))
            c_dict = dict(zip(cations, best_balance['c_combo']))
            
            sum_anions = best_balance['sum_anions']
            sum_cations = best_balance['sum_cations']
            
            # Собираем все ионы, у которых концентрация > 0
            all_ions_dict = {}
            for ion in anions:
                if a_dict[ion]['Conc'] > 0: all_ions_dict[ion] = a_dict[ion]
            for ion in cations:
                if ion != 'Na' and c_dict[ion]['Conc'] > 0: 
                    all_ions_dict[ion] = c_dict[ion]
            
            # Добавляем Натрий (измеренный или расчетный)
            all_ions_dict['Na'] = best_balance['na_item']
            
            # Формируем строки для Excel
            for ion, item in all_ions_dict.items():
                val_mg = item['Conc']
                val_meq = val_mg / weights[ion]
                eq_percent = (val_meq / (sum_anions + sum_cations)) * 100 # Доля иона в % экв.
                
                # Помечаем натрий, если он был рассчитан по разности
                ion_name = "Na (по разности)" if ion == 'Na' and best_balance['type'] == 'diff_Na' else ion
                
                final_results.append({
                    'Sample': sample_id, 'Ион': ion_name,
                    'Конц. (мг/л)': round(val_mg, 3), 'Эквив. вес': weights[ion],
                    'Конц. (мг-экв/л)': round(val_meq, 3), 'Эквив. %': round(eq_percent, 2),
                    'Gain': item['Gain'], 'Dilution': item['Dilution']
                })
            
            # Добавляем итоговые строки и пустую строку для разделения проб в Excel
            final_results.append({'Sample': sample_id, 'Ион': 'Σ Анионов', 'Конц. (мг/л)': '', 'Эквив. вес': '', 'Конц. (мг-экв/л)': round(sum_anions, 3), 'Эквив. %': '', 'Gain': '', 'Dilution': ''})
            final_results.append({'Sample': sample_id, 'Ион': 'Σ Катионов', 'Конц. (мг/л)': '', 'Эквив. вес': '', 'Конц. (мг-экв/л)': round(sum_cations, 3), 'Эквив. %': '', 'Gain': '', 'Dilution': ''})
            final_results.append({'Sample': sample_id, 'Ион': 'ОТКЛОНЕНИЕ, %', 'Конц. (мг/л)': '', 'Эквив. вес': '', 'Конц. (мг-экв/л)': round(min_error, 3), 'Эквив. %': '', 'Gain': '', 'Dilution': ''})
            final_results.append({}) # Пустая строка
            
        else:
            # --- ЛОГИКА ОШИБОК (Если Na везде отрицательный) ---
            # Берем максимально возможные анионы и минимальные катионы, чтобы понять масштаб бедствия
            max_anions = sum(max([i['Conc'] for i in available_data[ion]]) / weights[ion] if available_data[ion][0]['Conc'] != 0 else 0 for ion in anions)
            min_cations = sum(min([i['Conc'] for i in available_data[ion]]) / weights[ion] if available_data[ion][0]['Conc'] != 0 else 0 for ion in cations if ion != 'Na')
            
            deficit = max_anions - min_cations
            excess_ca = abs(deficit) * weights['Ca']
            
            # Ищем, каких важных ионов не хватает в пробе (игнорируя список исключений и Натрий)
            missing_ions = []
            for ion in anions + cations:
                if ion not in exclusions and ion != 'Na':
                    if available_data[ion][0]['Conc'] == 0:
                        missing_ions.append(ion)
            
            missing_str = ", ".join(missing_ions) if missing_ions else "Нет (все заданы)"
            
            # Формируем понятную рекомендацию для лаборанта
            if missing_ions:
                rec_text = f"ВНИМАНИЕ: В пробе отсутствуют данные по {missing_str}. Добавьте их или проверьте завышение Ca/Mg."
            else:
                rec_text = "Все анионы/катионы заданы. Проверьте завышение Ca²⁺/Mg²⁺ или нехватку анионов (кислые воды/органика)."
            
            error_results.append({
                'Sample': sample_id, 
                'Дефицит анионов (мг-экв/л)': round(deficit, 3),
                'Эквивалент лишнего Ca (мг/л)': round(excess_ca, 2),
                'Отсутствуют данные по:': missing_str,
                'Рекомендация': rec_text
            })

    # --- СОХРАНЕНИЕ В EXCEL ---
    if final_results:
        pd.DataFrame(final_results).to_excel(file_balance, index=False)
    if error_results:
        pd.DataFrame(error_results).to_excel(file_errors, index=False)

    # --- СТАТИСТИКА В КОНСОЛЬ ---
    total = len(samples)
    errors = len(error_results)
    success = total - errors
    
    print("\n" + "="*50)
    print("📊 СТАТИСТИКА ОБРАБОТКИ ДАННЫХ:")
    print(f"Всего проб в исходнике: {total}")
    print(f"✅ Успешно посчитано: {success} (сохранено в {file_balance})")
    print(f"❌ Попало в ошибки: {errors} (сохранено в {file_errors})")
    print("="*50)
    print("💡 Не забудьте проверить файл ошибок!")

# ЗАПУСК
if __name__ == "__main__":
    calculate_balance('clean_data.xlsx')

Загрузка очищенных данных...
Найдено проб: 72. Начинаем перебор комбинаций...

📊 СТАТИСТИКА ОБРАБОТКИ ДАННЫХ:
Всего проб в исходнике: 72
✅ Успешно посчитано: 40 (сохранено в Final_Balance_2026-11-05_15-55-25.xlsx)
❌ Попало в ошибки: 32 (сохранено в Errors_and_Recommendations_2026-11-05_15-55-25.xlsx)
💡 Не забудьте проверить файл ошибок!
